# 03 - Train Severity Model (Memory Optimized)

Train with 100K samples (float32) - fits in 12-13GB RAM

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/smart-ambulance-ml'
!pip install -q xgboost lightgbm joblib

import numpy as np
import pandas as pd
import gc
import joblib
import json
import time
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report
import warnings
warnings.filterwarnings('ignore')
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# Load data
print("Loading severity data...")
data = np.load(f'{PROJECT_DIR}/datasets/processed/severity_train.npz', allow_pickle=True)
X_train = data['X_train'].astype(np.float32)
X_val = data['X_val'].astype(np.float32)
X_test = data['X_test'].astype(np.float32)
y_train = data['y_train']
y_val = data['y_val']
y_test = data['y_test']
feature_names = data['feature_names'].tolist()

print(f"✓ Loaded: Train {X_train.shape}, Val {X_val.shape}, Test {X_test.shape}")
print(f"RAM usage: ~{X_train.nbytes / 1e9:.2f}GB for training data")

In [ ]:
print("\n="*60)
print("TRAINING SEVERITY MODELS")
print("="*60)

models = {
    'Random Forest': RandomForestClassifier(
        n_estimators=150, max_depth=12, min_samples_split=4,
        class_weight='balanced_subsample', random_state=42, n_jobs=-1
    ),
    'XGBoost': XGBClassifier(
        n_estimators=150, max_depth=6, learning_rate=0.05,
        subsample=0.85, colsample_bytree=0.85,
        eval_metric='mlogloss', random_state=42, n_jobs=-1,
        tree_method='hist'  # Memory efficient
    ),
    'LightGBM': LGBMClassifier(
        n_estimators=150, learning_rate=0.05, num_leaves=31,
        subsample=0.85, colsample_bytree=0.85,
        class_weight='balanced', random_state=42, n_jobs=-1, verbose=-1
    )
}

results = {}
trained_models = {}

for name, model in models.items():
    print(f"\nTraining {name}...")
    start = time.time()
    model.fit(X_train, y_train)
    trained_models[name] = model
    
    y_pred = model.predict(X_val)
    acc = accuracy_score(y_val, y_pred)
    f1 = f1_score(y_val, y_pred, average='weighted', zero_division=0)
    results[name] = {'accuracy': acc, 'f1_score': f1, 'time': time.time() - start}
    print(f"  ✓ Acc: {acc:.4f}, F1: {f1:.4f}")
    gc.collect()

print("\n" + pd.DataFrame(results).T.round(4).to_string())

In [ ]:
print("\n="*60)
print("TRAINING STACKING ENSEMBLE")
print("="*60)

base_estimators = [(name, model) for name, model in trained_models.items()]
meta_learner = LogisticRegression(max_iter=500, random_state=42, n_jobs=-1)

stacking_model = StackingClassifier(
    estimators=base_estimators,
    final_estimator=meta_learner,
    cv=2,  # Reduced from 3 to save memory
    stack_method='predict_proba',
    n_jobs=-1
)

print("Training stacking model...")
start = time.time()
stacking_model.fit(X_train, y_train)

y_pred_stack = stacking_model.predict(X_val)
acc_stack = accuracy_score(y_val, y_pred_stack)
f1_stack = f1_score(y_val, y_pred_stack, average='weighted', zero_division=0)

results['Stacking'] = {'accuracy': acc_stack, 'f1_score': f1_stack, 'time': time.time() - start}
trained_models['Stacking'] = stacking_model

print(f"✓ Stacking: Acc={acc_stack:.4f}, F1={f1_stack:.4f}")
gc.collect()

In [ ]:
print("\n="*60)
print("TEST EVALUATION")
print("="*60)

test_results = {}
for name, model in trained_models.items():
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
    test_results[name] = {'Accuracy': acc, 'F1': f1}

test_df = pd.DataFrame(test_results).T.sort_values('Accuracy', ascending=False)
print(test_df.round(4))

# Classification report for best
best_model = trained_models[test_df.index[0]]
y_pred_best = best_model.predict(X_test)
print(f"\nBest model: {test_df.index[0]}")
print(classification_report(y_test, y_pred_best, target_names=['ESI 1','ESI 2','ESI 3','ESI 4','ESI 5']))

In [ ]:
print("\n="*60)
print("SAVING MODELS")
print("="*60)

models_dir = f'{PROJECT_DIR}/models/severity'
os.makedirs(models_dir, exist_ok=True)

for name, model in trained_models.items():
    filename = name.lower().replace(' ', '_') + '.joblib'
    joblib.dump(model, f'{models_dir}/{filename}')
    print(f"✓ Saved {name}")

# Registry
registry = {'severity': {'models': list(trained_models.keys()),
             'best_model': test_df.index[0],
             'test_accuracy': test_results[test_df.index[0]]['Accuracy'],
             'test_f1': test_results[test_df.index[0]]['F1'],
             'feature_count': len(feature_names)}}

with open(f'{PROJECT_DIR}/models/model_registry.json', 'w') as f:
    json.dump(registry, f, indent=2)

print(f"\n✓ Training complete! Best: {test_df.index[0]}")
print(f"  Accuracy: {test_results[test_df.index[0]]['Accuracy']:.4f}")
gc.collect()